# Approach 2 — IL + RL + Reward Shaping

Loads BC weights (optional), then fine-tunes with **PPO + dense reward**.

Dense reward: `r = 0.3 × r_reach + 1.0 × r_lid + r_success`

**Output:** `results/ppo_rs_model.zip`, `results/ppo_rs_eval.json`

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import torch
import matplotlib.pyplot as plt
from common import RESULTS_DIR

print(f'Project root : {PROJECT_ROOT}')
print(f'Results dir  : {RESULTS_DIR}')
print(f'Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')

bc_model_path = RESULTS_DIR / 'bc_model.pt'
if bc_model_path.exists():
    print(f'BC model     : {bc_model_path} ✓')
else:
    print('BC model     : NOT FOUND — will train PPO from scratch (no BC init)')

Project root : /home/contente/Desktop/RoboCasa-Project
Results dir  : /home/contente/Desktop/RoboCasa-Project/results
Device       : cuda
BC model     : /home/contente/Desktop/RoboCasa-Project/results/bc_model.pt ✓


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Hyperparameters

In [2]:
# ── Edit these before running ─────────────────────────────────────────────
TOTAL_TIMESTEPS = 500_000
N_ENVS          = 4
LEARNING_RATE   = 3e-4
N_STEPS         = 2048       # steps per env per PPO update
BATCH_SIZE      = 256
EVAL_FREQ       = 10_000     # timesteps between EvalCallback evaluations
N_EVAL_EPISODES = 20         # evaluation episodes after training
USE_BC_INIT     = True       # set False to train from random init
# ─────────────────────────────────────────────────────────────────────────

bc_arg = []
if USE_BC_INIT and bc_model_path.exists():
    bc_arg = ['--bc-model', str(bc_model_path)]
    print('Will transfer BC weights to PPO.')
elif USE_BC_INIT:
    print('Warning: USE_BC_INIT=True but bc_model.pt not found. Run 01_BC.ipynb first.')

Will transfer BC weights to PPO.


## Train

In [ ]:
import train_il_rl_rs
import importlib; importlib.reload(train_il_rl_rs)

train_il_rl_rs.main([
    '--timesteps', str(TOTAL_TIMESTEPS),
    '--n-envs',    str(N_ENVS),
    '--lr',        str(LEARNING_RATE),
    '--n-steps',   str(N_STEPS),
    '--batch-size',str(BATCH_SIZE),
    '--eval-freq', str(EVAL_FREQ),
    '--eval-ep',   str(N_EVAL_EPISODES),
    '--output',    str(RESULTS_DIR),
] + bc_arg)

[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly (e.g. pip install mink==0.0.5), otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)
14:47:52  INFO      Building 4 training environment(s) …
[robosuite INFO] Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)
14:47:52  INFO      Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json


[robosuite INFO] Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)
14:47:58  INFO      Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json
[robosuite INFO] Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)
14:48:05  INFO      Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json
[robosuite INFO] Loading controller configuration from: /home/contente/Desktop/RoboCasa-Project/deps/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)
14:48:10  INFO      Loadi

Using cuda device


/home/contente/Desktop/RoboCasa-Project/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
14:48:23  INFO      Loading BC weights from /home/contente/Desktop/RoboCasa-Project/results/bc_model.pt …
14:48:23  INFO      [BC→PPO] Transferred 6 / 6 parameter tensors.
14:48:23  INFO      BC → PPO weight transfer complete.
14:48:23  INFO      Training PPO for 500000 timesteps …


-----------------------------
| time/              |      |
|    fps             | 15   |
|    iterations      | 1    |
|    time_elapsed    | 519  |
|    total_timesteps | 8192 |
-----------------------------


/home/contente/Desktop/RoboCasa-Project/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=10000, episode_reward=0.00 +/- 0.00
Episode length: 500.00 +/- 0.00
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 500         |
|    mean_reward          | 9.13e-06    |
| time/                   |             |
|    total_timesteps      | 10000       |
| train/                  |             |
|    approx_kl            | 0.086916395 |
|    clip_fraction        | 0.544       |
|    clip_range           | 0.2         |
|    entropy_loss         | -17         |
|    explained_variance   | -13.6       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0811     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0456     |
|    std                  | 1           |
|    value_loss           | 0.0356      |
-----------------------------------------
New best mean reward!
------------------------------
| time/              |       |
|    fps             | 12    |


## Evaluation Results

In [ ]:
eval_file = RESULTS_DIR / 'ppo_rs_eval.json'
if not eval_file.exists():
    print('ppo_rs_eval.json not found.')
else:
    metrics = json.loads(eval_file.read_text())
    print('PPO-RS Evaluation Results')
    print('─' * 35)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k:<28}: {v:.4f}')
        else:
            print(f'  {k:<28}: {v}')

## Saved Artefacts

| File | Description |
|------|-------------|
| `results/ppo_rs_model.zip` | Final PPO checkpoint |
| `results/ppo_rs_best/` | Best checkpoint (by EvalCallback) |
| `results/ppo_rs_eval.json` | Evaluation metrics |

These are loaded automatically by `05_Analysis.ipynb`.